In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


In [2]:
trades = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")


In [3]:
trades["date"] = pd.to_datetime(trades["Timestamp"]).dt.date
sentiment["date"] = pd.to_datetime(sentiment["timestamp"]).dt.date


In [5]:
trades.columns.tolist()


['Account',
 'Coin',
 'Execution Price',
 'Size Tokens',
 'Size USD',
 'Side',
 'Timestamp IST',
 'Start Position',
 'Direction',
 'Closed PnL',
 'Transaction Hash',
 'Order ID',
 'Crossed',
 'Fee',
 'Trade ID',
 'Timestamp',
 'date']

In [7]:
daily_metrics = (
    trades.groupby(["Account", "date"])
    .agg(
        daily_pnl=("Closed PnL", "sum"),
        trade_count=("Trade ID", "count"),
        avg_trade_size=("Size USD", "mean"),
        long_ratio=("Direction", lambda x: (x == "Long").mean())
    )
    .reset_index()
)

dashboard_df = daily_metrics.merge(
    sentiment[["date", "classification"]],
    on="date",
    how="left"
)


In [8]:
dashboard_df.head()


,Account,date,daily_pnl,trade_count,avg_trade_size,long_ratio,classification
0,0x083384f897ee0f19899168e3b1bec365f52a9012,1970-01-01,1.600230e+06,3818,16159.576734,0.0,Fear
1,0x083384f897ee0f19899168e3b1bec365f52a9012,1970-01-01,1.600230e+06,3818,16159.576734,0.0,Extreme Fear
2,0x083384f897ee0f19899168e3b1bec365f52a9012,1970-01-01,1.600230e+06,3818,16159.576734,0.0,Fear
3,0x083384f897ee0f19899168e3b1bec365f52a9012,1970-01-01,1.600230e+06,3818,16159.576734,0.0,Extreme Fear
4,0x083384f897ee0f19899168e3b1bec365f52a9012,1970-01-01,1.600230e+06,3818,16159.576734,0.0,Extreme Fear


In [9]:
account_widget = widgets.Dropdown(
    options=sorted(dashboard_df["Account"].unique()),
    description="Account:"
)

metric_widget = widgets.Dropdown(
    options=["daily_pnl", "trade_count", "avg_trade_size", "long_ratio"],
    description="Metric:"
)


In [10]:
def plot_dashboard(account, metric):
    data = dashboard_df[dashboard_df["Account"] == account]

    plt.figure()
    plt.plot(data["date"], data[metric], marker="o")
    plt.title(f"{metric.replace('_',' ').title()} | Account {account}")
    plt.xlabel("Date")
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()


In [11]:
dashboard = widgets.interactive(
    plot_dashboard,
    account=account_widget,
    metric=metric_widget
)

display(dashboard)


interactive(children=(Dropdown(description='Account:', options=('0x083384f897ee0f19899168e3b1bec365f52a9012', …

In [12]:
dashboard_df[dashboard_df["Account"] == "0x6d6a4b953f202f8df5bed40692e7fd865318264a"]["date"].nunique()


1